# NOTEBOOK 02: EXPLORATORY DATA ANALYSIS (EDA)

## Air Quality Monitoring Analysis - European Cities

**Fokus: Professional Visualizations, Color Harmony, Design Excellence**

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# SETUP & PATHS

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [9]:
import os
BASE_PATH = '/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities'
CLEANED_DATA_PATH = f'{BASE_PATH}/data/cleaned'
RESULTS_EDA = f'{BASE_PATH}/results/eda'

os.makedirs(RESULTS_EDA, exist_ok=True)

print("✅ Setup complete")

✅ Setup complete


# LOAD CLEANED DATA

In [ ]:
df_all = pd.read_csv(f'{CLEANED_DATA_PATH}/all_cities_cleaned_with_anomalies.csv')
df_all['date'] = pd.to_datetime(df_all['date'])

print(f"✅ Data loaded: {len(df_all):,} rows, {len(df_all.columns)} columns")
print(f"   Date range: {df_all['date'].min().date()} to {df_all['date'].max().date()}")
print(f"   Cities: {df_all['city'].unique()}")

# DEFINE COLOR PALETTE (PROFESSIONAL & HARMONIOUS)

In [ ]:
# Primary color palette (untuk cities)
COLORS_CITIES = {
    'Ancona': '#3498DB',    # Professional Blue
    'Athens': '#E74C3C',    # Warm Red
    'Zaragoza': '#2ECC71'   # Fresh Green
}

# Pollutants color palette
COLORS_POLLUTANTS = {
    'pm25': '#FF6B6B',      # PM2.5 - Coral Red
    'pm10': '#FFA07A',      # PM10 - Light Salmon
    'no2': '#9B59B6',       # NO2 - Purple
    'o3': '#F39C12'         # O3 - Orange
}

# Background & text
BG_COLOR = 'rgba(245, 247, 250, 1)'
GRID_COLOR = 'rgba(230, 236, 245, 0.5)'
TEXT_COLOR = '#2C3E50'
ACCENT_COLOR = '#1f77b4'

print(f"✅ Color palettes defined")
print(f"   Cities: {list(COLORS_CITIES.keys())}")
print(f"   Pollutants: {list(COLORS_POLLUTANTS.keys())}")

# SECTION 1: UNIVARIATE ANALYSIS - POLLUTANTS DISTRIBUTIONS

In [ ]:
pollutants = ['pm25', 'pm10', 'no2', 'o3']
pollutant_names = ['PM2.5', 'PM10', 'NO₂', 'O₃']

# Create distribution plots with subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{name} Distribution' for name in pollutant_names],
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
           [{'secondary_y': False}, {'secondary_y': False}]]
)

for idx, (pollutant, name, color) in enumerate(zip(pollutants, pollutant_names,
                                                     ['#FF6B6B', '#FFA07A', '#9B59B6', '#F39C12']), 1):
    row = ((idx - 1) // 2) + 1
    col = ((idx - 1) % 2) + 1

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=df_all[pollutant],
            nbinsx=50,
            name=name,
            marker=dict(color=color, opacity=0.7, line=dict(color='white', width=0.5)),
            hovertemplate=f'<b>{name}</b><br>Range: %{{x}}<br>Count: %{{y}}<extra></extra>',
            showlegend=False
        ),
        row=row, col=col
    )

    # Add KDE overlay menggunakan histogram density
    fig.update_xaxes(title_text=f'{name} (µg/m³)', row=row, col=col)
    fig.update_yaxes(title_text='Frequency', row=row, col=col)

fig.update_layout(
    title_text='Pollutants Distribution Analysis',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=700,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/01_pollutants_distributions.html')
print("✅ Saved: 01_pollutants_distributions.html")
fig.show()

✅ Saved: 01_pollutants_distributions.html
Buffered data was truncated after reaching the output size limit.

# SECTION 2: UNIVARIATE ANALYSIS - METEOROLOGICAL DATA

In [ ]:
meteorology = ['temperature', 'relative_humidity', 'wind_u', 'wind_v']
meteorology_names = ['Temperature', 'Relative Humidity', 'Wind Speed (U)', 'Wind Speed (V)']
meteorology_colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{name} Distribution' for name in meteorology_names],
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'box'}, {'type': 'box'}]]
)

for idx, (var, name, color) in enumerate(zip(meteorology, meteorology_names, meteorology_colors), 1):
    row = ((idx - 1) // 2) + 1
    col = ((idx - 1) % 2) + 1

    fig.add_trace(
        go.Box(
            y=df_all[var],
            name=name,
            marker=dict(color=color),
            boxmean='sd',
            hovertemplate=f'<b>{name}</b><br>Value: %{{y:.2f}}<extra></extra>',
            showlegend=False,
            line=dict(width=2)
        ),
        row=row, col=col
    )

    fig.update_yaxes(title_text=name, row=row, col=col)

fig.update_layout(
    title_text='Meteorological Variables Distribution',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=700,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/02_meteorology_distributions.html')
print("✅ Saved: 02_meteorology_distributions.html")
fig.show()

✅ Saved: 02_meteorology_distributions.html
Buffered data was truncated after reaching the output size limit.

# SECTION 3: COMPARATIVE ANALYSIS - CITIES COMPARISON

In [ ]:
cities = df_all['city'].unique()

# Prepare data untuk comparison
comparison_data = []
for city in cities:
    df_city = df_all[df_all['city'] == city]
    comparison_data.append({
        'City': city,
        'PM2.5': df_city['pm25'].mean(),
        'PM10': df_city['pm10'].mean(),
        'NO₂': df_city['no2'].mean(),
        'O₃': df_city['o3'].mean(),
        'Temperature': df_city['temperature'].mean(),
        'Humidity': df_city['relative_humidity'].mean()
    })

comparison_df = pd.DataFrame(comparison_data)

# Horizontal bar chart untuk pollutants
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=['PM2.5 Comparison', 'PM10 Comparison', 'NO₂ Comparison', 'O₃ Comparison'],
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

pollutants_to_plot = ['PM2.5', 'PM10', 'NO₂', 'O₃']
pollutant_colors_list = ['#FF6B6B', '#FFA07A', '#9B59B6', '#F39C12']

for col_idx, (pollutant, color) in enumerate(zip(pollutants_to_plot, pollutant_colors_list), 1):
    values = comparison_df[pollutant].values

    fig.add_trace(
        go.Bar(
            x=comparison_df['City'],
            y=values,
            name=pollutant,
            marker=dict(color=color, line=dict(color='white', width=2)),
            text=[f'{v:.1f}' for v in values],
            textposition='outside',
            hovertemplate=f'<b>%{{x}}</b><br>{pollutant}: %{{y:.2f}} µg/m³<extra></extra>',
            showlegend=False
        ),
        row=1, col=col_idx
    )

    fig.update_yaxes(title_text=f'{pollutant} (µg/m³)', row=1, col=col_idx)

fig.update_layout(
    title_text='Pollutants Mean Levels - Cities Comparison',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=500,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/03_cities_pollutants_comparison.html')
print("✅ Saved: 03_cities_pollutants_comparison.html")
fig.show()

✅ Saved: 03_cities_pollutants_comparison.html


# SECTION 4: CORRELATION ANALYSIS - HEATMAPS PER CITY

In [ ]:
# Select columns untuk correlation
corr_columns = ['pm25', 'pm10', 'no2', 'o3', 'temperature', 'relative_humidity', 'wind_u', 'wind_v', 'precipitation']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Ancona Correlations', 'Athens Correlations', 'Zaragoza Correlations'],
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}, {'type': 'heatmap'}]]
)

cities_list = ['Ancona', 'Athens', 'Zaragoza']
heatmap_colors = ['Blues', 'Reds', 'Greens']

In [ ]:
for col_idx, (city, colorscale) in enumerate(zip(cities_list, heatmap_colors), 1):
    df_city = df_all[df_all['city'] == city]
    corr_matrix = df_city[corr_columns].corr()

    fig.add_trace(
        go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.columns,
            colorscale=colorscale,
            zmid=0,
            zmin=-1,
            zmax=1,
            colorbar=dict(title='Correlation', x=0.35 + (col_idx-1)*0.3),
            hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>Correlation: %{z:.2f}<extra></extra>',
            showscale=(col_idx==1)
        ),
        row=1, col=col_idx
    )

    fig.update_xaxes(tickangle=45, row=1, col=col_idx)

fig.update_layout(
    title_text='Correlation Matrix - All Cities',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=600,
    paper_bgcolor='white',
    font=dict(size=10, color=TEXT_COLOR),
    margin=dict(l=100, r=100, t=100, b=150)
)

fig.write_html(f'{RESULTS_EDA}/04_correlation_heatmaps.html')
print("✅ Saved: 04_correlation_heatmaps.html")
fig.show()

✅ Saved: 04_correlation_heatmaps.html


# SECTION 5: STATISTICAL SUMMARY

In [ ]:
# Create comprehensive statistics table
stats_dict = {
    'Statistic': ['Count', 'Mean', 'Std Dev', 'Min', '25%', 'Median', '75%', 'Max', 'Skewness', 'Kurtosis']
}

for city in cities_list:
    df_city = df_all[df_all['city'] == city]

    stats_dict[f'{city} (PM2.5)'] = [
        len(df_city),
        df_city['pm25'].mean(),
        df_city['pm25'].std(),
        df_city['pm25'].min(),
        df_city['pm25'].quantile(0.25),
        df_city['pm25'].median(),
        df_city['pm25'].quantile(0.75),
        df_city['pm25'].max(),
        stats.skew(df_city['pm25'].dropna()),
        stats.kurtosis(df_city['pm25'].dropna())
    ]

stats_df = pd.DataFrame(stats_dict)

print("\n📊 DESCRIPTIVE STATISTICS - PM2.5:")
print(stats_df.to_string(index=False))


📊 DESCRIPTIVE STATISTICS - PM2.5:
Statistic  Ancona (PM2.5)  Athens (PM2.5)  Zaragoza (PM2.5)
    Count   417626.000000    1.726464e+06          238336.0
     Mean       13.013333    1.507474e+01               NaN
  Std Dev        7.787227    1.506120e+01               NaN
      Min        0.000000    0.000000e+00               NaN
      25%        7.871392    8.000000e+00               NaN
   Median       11.192256    1.165063e+01               NaN
      75%       16.146508    1.776377e+01               NaN
      Max      200.000000    1.748715e+03               NaN
 Skewness        2.480792    2.457189e+01               NaN
 Kurtosis       18.573667    2.214953e+03               NaN


In [ ]:
# Save statistics table
stats_df.to_csv(f'{RESULTS_EDA}/statistics_summary.csv', index=False)
print(f"\n✅ Saved: statistics_summary.csv")


✅ Saved: statistics_summary.csv


In [ ]:
# Visualisasi statistics dengan Plotly table
fig = go.Figure(
    data=[go.Table(
        header=dict(
            values=list(stats_df.columns),
            fill_color=ACCENT_COLOR,
            align='center',
            font=dict(color='white', size=12)
        ),
        cells=dict(
            values=[stats_df[col] for col in stats_df.columns],
            fill_color='lavender',
            align='center',
            font=dict(color=TEXT_COLOR, size=11),
            height=25
        )
    )]
)

fig.update_layout(
    title_text='Descriptive Statistics - PM2.5 per City',
    title_font_size=20,
    title_font_color=ACCENT_COLOR,
    height=400,
    margin=dict(l=20, r=20, t=80, b=20)
)

fig.write_html(f'{RESULTS_EDA}/05_statistics_table.html')
print("✅ Saved: 05_statistics_table.html")
fig.show()

✅ Saved: 05_statistics_table.html


# SECTION 6: AQI CATEGORIZATION & ANALYSIS

In [ ]:
# Define AQI categories berdasarkan WHO & ISPU standards
def categorize_aqi(pm25_value):
    """Kategorisasi AQI berdasarkan PM2.5 level (WHO standards)"""
    if pm25_value <= 15:
        return 'Good'
    elif pm25_value <= 35:
        return 'Moderate'
    elif pm25_value <= 75:
        return 'Unhealthy for Sensitive'
    elif pm25_value <= 115:
        return 'Unhealthy'
    else:
        return 'Hazardous'

df_all['aqi_category'] = df_all['pm25'].apply(categorize_aqi)

# AQI distribution per city
aqi_counts = df_all.groupby(['city', 'aqi_category']).size().unstack(fill_value=0)
aqi_counts = aqi_counts.reindex(['Good', 'Moderate', 'Unhealthy for Sensitive', 'Unhealthy', 'Hazardous'], fill_value=0)

print("\n📊 AQI DISTRIBUTION PER CITY:")
print(aqi_counts)


📊 AQI DISTRIBUTION PER CITY:
aqi_category             Good  Hazardous  Moderate  Unhealthy  \
city                                                            
Good                        0          0         0          0   
Moderate                    0          0         0          0   
Unhealthy for Sensitive     0          0         0          0   
Unhealthy                   0          0         0          0   
Hazardous                   0          0         0          0   

aqi_category             Unhealthy for Sensitive  
city                                              
Good                                           0  
Moderate                                       0  
Unhealthy for Sensitive                        0  
Unhealthy                                      0  
Hazardous                                      0  


In [ ]:
# Calculate percentage
aqi_pct = aqi_counts.div(aqi_counts.sum(axis=1), axis=0) * 100

# Visualisasi AQI distribution
aqi_colors_map = {
    'Good': '#2ECC71',
    'Moderate': '#F39C12',
    'Unhealthy for Sensitive': '#E67E22',
    'Unhealthy': '#E74C3C',
    'Hazardous': '#C0392B'
}

fig = go.Figure()

for category in ['Good', 'Moderate', 'Unhealthy for Sensitive', 'Unhealthy', 'Hazardous']:
    fig.add_trace(
        go.Bar(
            x=aqi_counts.index,
            y=aqi_counts[category],
            name=category,
            marker=dict(color=aqi_colors_map[category], line=dict(color='white', width=1)),
            hovertemplate='<b>%{x}</b><br>' + category + ': %{y} days<extra></extra>'
        )
    )

fig.update_layout(
    title_text='Air Quality Index (AQI) Distribution - Days per Category',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    xaxis_title='City',
    yaxis_title='Number of Days',
    barmode='stack',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=12, color=TEXT_COLOR),
    legend=dict(orientation='v', yanchor='top', y=0.99, xanchor='right', x=0.99),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/06_aqi_distribution.html')
print("\n✅ Saved: 06_aqi_distribution.html")
fig.show()


✅ Saved: 06_aqi_distribution.html


# SECTION 7: COMPLIANCE ANALYSIS (WHO STANDARDS)

In [ ]:
# WHO guidelines
WHO_GUIDELINES = {
    'pm25': 15,      # µg/m³ daily mean
    'pm10': 45,      # µg/m³ daily mean
    'no2': 40,       # µg/m³ annual mean
    'o3': 100        # µg/m³ 8-hour mean
}

compliance_data = []
for city in cities_list:
    df_city = df_all[df_all['city'] == city]

    compliance_data.append({
        'City': city,
        'PM2.5 Compliance %': ((df_city['pm25'] <= WHO_GUIDELINES['pm25']).sum() / len(df_city)) * 100,
        'PM10 Compliance %': ((df_city['pm10'] <= WHO_GUIDELINES['pm10']).sum() / len(df_city)) * 100,
        'NO₂ Compliance %': ((df_city['no2'] <= WHO_GUIDELINES['no2']).sum() / len(df_city)) * 100,
        'O₃ Compliance %': ((df_city['o3'] <= WHO_GUIDELINES['o3']).sum() / len(df_city)) * 100
    })

compliance_df = pd.DataFrame(compliance_data)

print("\n📊 WHO GUIDELINES COMPLIANCE (% days within limits):")
print(compliance_df.to_string(index=False))


📊 WHO GUIDELINES COMPLIANCE (% days within limits):
    City  PM2.5 Compliance %  PM10 Compliance %  NO₂ Compliance %  O₃ Compliance %
  Ancona           70.516922          97.726914         97.994138        94.383491
  Athens           66.545147          91.891925         85.177913        87.282446
Zaragoza            0.000000          96.401299         93.115182        96.775980


In [ ]:
# Visualisasi compliance
fig = go.Figure()

for idx, city in enumerate(cities_list):
    values = [
        compliance_df.loc[compliance_df['City'] == city, 'PM2.5 Compliance %'].values[0],
        compliance_df.loc[compliance_df['City'] == city, 'PM10 Compliance %'].values[0],
        compliance_df.loc[compliance_df['City'] == city, 'NO₂ Compliance %'].values[0],
        compliance_df.loc[compliance_df['City'] == city, 'O₃ Compliance %'].values[0]
    ]

    fig.add_trace(
        go.Bar(
            x=['PM2.5', 'PM10', 'NO₂', 'O₃'],
            y=values,
            name=city,
            marker=dict(color=COLORS_CITIES[city], opacity=0.8, line=dict(color='white', width=2)),
            text=[f'{v:.1f}%' for v in values],
            textposition='outside',
            hovertemplate='<b>%{x}</b> (%{fullData.name})<br>Compliance: %{y:.1f}%<extra></extra>'
        )
    )

fig.add_hline(y=100, line_dash='dash', line_color='green', annotation_text='WHO Target',
              annotation_position='right')

fig.update_layout(
    title_text='WHO Guidelines Compliance - Percentage of Days within Limits',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    xaxis_title='Pollutant',
    yaxis_title='Compliance %',
    barmode='group',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=12, color=TEXT_COLOR),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/07_compliance_analysis.html')
print("\n✅ Saved: 07_compliance_analysis.html")
fig.show()


✅ Saved: 07_compliance_analysis.html


In [ ]:




# ============================================================================
# SECTION 8: VIOLATIONS COUNT & SEVERITY
# ============================================================================

print("\n" + "="*80)
print("SECTION 8: VIOLATIONS COUNT & SEVERITY")
print("="*80)

violations_data = []
for city in cities_list:
    df_city = df_all[df_all['city'] == city]

    pm25_violations = (df_city['pm25'] > WHO_GUIDELINES['pm25']).sum()
    pm10_violations = (df_city['pm10'] > WHO_GUIDELINES['pm10']).sum()
    no2_violations = (df_city['no2'] > WHO_GUIDELINES['no2']).sum()
    o3_violations = (df_city['o3'] > WHO_GUIDELINES['o3']).sum()

    violations_data.append({
        'City': city,
        'PM2.5 Violations': pm25_violations,
        'PM10 Violations': pm10_violations,
        'NO₂ Violations': no2_violations,
        'O₃ Violations': o3_violations
    })

violations_df = pd.DataFrame(violations_data)

print("\n📊 VIOLATIONS COUNT (days exceeding WHO limits):")
print(violations_df.to_string(index=False))

# Visualisasi violations
fig = go.Figure()

for idx, city in enumerate(cities_list):
    violations = [
        violations_df.loc[violations_df['City'] == city, 'PM2.5 Violations'].values[0],
        violations_df.loc[violations_df['City'] == city, 'PM10 Violations'].values[0],
        violations_df.loc[violations_df['City'] == city, 'NO₂ Violations'].values[0],
        violations_df.loc[violations_df['City'] == city, 'O₃ Violations'].values[0]
    ]

    fig.add_trace(
        go.Bar(
            x=['PM2.5', 'PM10', 'NO₂', 'O₃'],
            y=violations,
            name=city,
            marker=dict(color=COLORS_CITIES[city], opacity=0.8, line=dict(color='white', width=2)),
            text=violations,
            textposition='outside',
            hovertemplate='<b>%{x}</b> (%{fullData.name})<br>Violations: %{y} days<extra></extra>'
        )
    )

fig.update_layout(
    title_text='Violations of WHO Guidelines - Days Exceeding Limits',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    xaxis_title='Pollutant',
    yaxis_title='Number of Violation Days',
    barmode='group',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=12, color=TEXT_COLOR),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_EDA}/08_violations_analysis.html')
print("\n✅ Saved: 08_violations_analysis.html")
fig.show()

# ============================================================================
# SECTION 9: RELATIONSHIP BETWEEN POLLUTANTS
# ============================================================================

print("\n" + "="*80)
print("SECTION 9: POLLUTANT RELATIONSHIPS")
print("="*80)

# Create scatter matrix untuk pollutant relationships
fig = px.scatter_matrix(
    df_all[['pm25', 'pm10', 'no2', 'o3', 'city']],
    dimensions=['pm25', 'pm10', 'no2', 'o3'],
    color='city',
    color_discrete_map=COLORS_CITIES,
    labels={'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO₂', 'o3': 'O₃'},
    title='Pollutants Relationship Matrix',
    opacity=0.6,
    hover_data=['city']
)

fig.update_traces(
    diagonal_visible=False,
    showupperhalf=False,
    marker=dict(line=dict(color='white', width=0.5))
)

fig.update_layout(
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=800,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    font=dict(size=11, color=TEXT_COLOR),
    hovermode='closest'
)

fig.write_html(f'{RESULTS_EDA}/09_pollutants_relationships.html')
print("✅ Saved: 09_pollutants_relationships.html")
fig.show()

# ============================================================================
# SECTION 10: METEOROLOGICAL IMPACT ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("SECTION 10: METEOROLOGICAL IMPACT ON POLLUTANTS")
print("="*80)

# Analyze correlation antara meteorology dan pollutants
meteorology_impact = []
for city in cities_list:
    df_city = df_all[df_all['city'] == city]

    correlations = {
        'City': city,
        'PM2.5 vs Temp': df_city['pm25'].corr(df_city['temperature']),
        'PM2.5 vs Humidity': df_city['pm25'].corr(df_city['relative_humidity']),
        'PM2.5 vs Wind (U)': df_city['pm25'].corr(df_city['wind_u']),
        'NO₂ vs Temp': df_city['no2'].corr(df_city['temperature']),
        'NO₂ vs Wind (U)': df_city['no2'].corr(df_city['wind_u'])
    }
    meteorology_impact.append(correlations)

meteorology_df = pd.DataFrame(meteorology_impact)

print("\n📊 METEOROLOGICAL IMPACT - CORRELATIONS:")
print(meteorology_df.to_string(index=False))

# Visualisasi meteorological impact
fig = go.Figure()

for city in cities_list:
    city_data = meteorology_df[meteorology_df['City'] == city].iloc[0]

    fig.add_trace(
        go.Bar(
            x=['PM2.5 vs\nTemperature', 'PM2.5 vs\nHumidity', 'PM2.5 vs\nWind (U)',
               'NO₂ vs\nTemperature', 'NO₂ vs\nWind (U)'],
            y=[city_data['PM2.5 vs Temp'], city_data['PM2.5 vs Humidity'],
               city_data['PM2.5 vs Wind (U)'], city_data['NO₂ vs Temp'],
               city_data['NO₂ vs Wind (U)']],
            name=city,
            marker=dict(color=COLORS_CITIES[city], opacity=0.8, line=dict(color='white', width=2)),
            text=[f'{x:.2f}' for x in [city_data['PM2.5 vs Temp'], city_data['PM2.5 vs Humidity'],
                                        city_data['PM2.5 vs Wind (U)'], city_data['NO₂ vs Temp'],
                                        city_data['NO₂ vs Wind (U)']]],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Correlation: %{y:.3f}<extra></extra>'
        )
    )

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)

fig.update_layout(
    title_text='Meteorological Impact on Pollutants - Correlation Analysis',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    yaxis_title='Correlation Coefficient',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=12, color=TEXT_COLOR),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=80, r=50, t=100, b=100)
)

fig.write_html(f'{RESULTS_EDA}/10_meteorological_impact.html')
print("\n✅ Saved: 10_meteorological_impact.html")
fig.show()

# ============================================================================
# SECTION 11: SAVE COMPREHENSIVE EDA SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("SECTION 11: SAVE EDA SUMMARY REPORT")
print("="*80)

# Create comprehensive summary report
summary_report = f"""
# AIR QUALITY MONITORING - EXPLORATORY DATA ANALYSIS REPORT
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## DATASET OVERVIEW
- Total Records: {len(df_all):,}
- Date Range: {df_all['date'].min().date()} to {df_all['date'].max().date()}
- Cities Analyzed: {', '.join(cities_list)}
- Variables: {len(df_all.columns)}

## KEY FINDINGS

### 1. POLLUTANTS ANALYSIS
#### PM2.5 (Fine Particulate Matter)
- Overall Mean: {df_all['pm25'].mean():.2f} µg/m³
- WHO Guideline: 15 µg/m³
- Compliance Rate: {((df_all['pm25'] <= 15).sum() / len(df_all) * 100):.1f}%
- Days in Violation: {(df_all['pm25'] > 15).sum()}

#### PM10 (Coarse Particulate Matter)
- Overall Mean: {df_all['pm10'].mean():.2f} µg/m³
- WHO Guideline: 45 µg/m³
- Compliance Rate: {((df_all['pm10'] <= 45).sum() / len(df_all) * 100):.1f}%
- Days in Violation: {(df_all['pm10'] > 45).sum()}

#### NO₂ (Nitrogen Dioxide)
- Overall Mean: {df_all['no2'].mean():.2f} µg/m³
- WHO Guideline: 40 µg/m³
- Compliance Rate: {((df_all['no2'] <= 40).sum() / len(df_all) * 100):.1f}%
- Days in Violation: {(df_all['no2'] > 40).sum()}

#### O₃ (Ozone)
- Overall Mean: {df_all['o3'].mean():.2f} µg/m³
- WHO Guideline: 100 µg/m³
- Compliance Rate: {((df_all['o3'] <= 100).sum() / len(df_all) * 100):.1f}%
- Days in Violation: {(df_all['o3'] > 100).sum()}

### 2. CITY COMPARISON
"""

for city in cities_list:
    df_city = df_all[df_all['city'] == city]
    summary_report += f"""
#### {city}
- Records: {len(df_city):,}
- PM2.5 Mean: {df_city['pm25'].mean():.2f} µg/m³
- PM10 Mean: {df_city['pm10'].mean():.2f} µg/m³
- NO₂ Mean: {df_city['no2'].mean():.2f} µg/m³
- O₃ Mean: {df_city['o3'].mean():.2f} µg/m³
- Temperature Mean: {df_city['temperature'].mean():.2f}°C
- Humidity Mean: {df_city['relative_humidity'].mean():.1f}%
"""

summary_report += f"""

### 3. AIR QUALITY STATUS
- Good Days (AQI): {(df_all['aqi_category'] == 'Good').sum()} ({(df_all['aqi_category'] == 'Good').sum() / len(df_all) * 100:.1f}%)
- Moderate Days: {(df_all['aqi_category'] == 'Moderate').sum()} ({(df_all['aqi_category'] == 'Moderate').sum() / len(df_all) * 100:.1f}%)
- Unhealthy for Sensitive Days: {(df_all['aqi_category'] == 'Unhealthy for Sensitive').sum()} ({(df_all['aqi_category'] == 'Unhealthy for Sensitive').sum() / len(df_all) * 100:.1f}%)
- Unhealthy Days: {(df_all['aqi_category'] == 'Unhealthy').sum()} ({(df_all['aqi_category'] == 'Unhealthy').sum() / len(df_all) * 100:.1f}%)
- Hazardous Days: {(df_all['aqi_category'] == 'Hazardous').sum()} ({(df_all['aqi_category'] == 'Hazardous').sum() / len(df_all) * 100:.1f}%)

### 4. METEOROLOGICAL INSIGHTS
- Temperature Range: {df_all['temperature'].min():.1f}°C to {df_all['temperature'].max():.1f}°C
- Mean Temperature: {df_all['temperature'].mean():.2f}°C
- Humidity Range: {df_all['relative_humidity'].min():.1f}% to {df_all['relative_humidity'].max():.1f}%
- Mean Humidity: {df_all['relative_humidity'].mean():.2f}%

## VISUALIZATIONS GENERATED
1. 01_pollutants_distributions.html - Distribution analysis for all pollutants
2. 02_meteorology_distributions.html - Box plots for meteorological variables
3. 03_cities_pollutants_comparison.html - Comparative analysis across cities
4. 04_correlation_heatmaps.html - Correlation matrices per city
5. 05_statistics_table.html - Descriptive statistics
6. 06_aqi_distribution.html - AQI categories distribution
7. 07_compliance_analysis.html - WHO guidelines compliance
8. 08_violations_analysis.html - Violations count analysis
9. 09_pollutants_relationships.html - Scatter matrix of pollutant relationships
10. 10_meteorological_impact.html - Impact of meteorology on pollutants

## NEXT STEPS
1. Time Series Analysis - Trend detection and seasonality
2. Geospatial Analysis - Spatial distribution and hotspots
3. Pattern Analysis - Temporal and meteorological patterns
4. Anomaly Detection - Outlier and anomaly identification
5. Machine Learning Forecasting - Predictive modeling
"""

# Save report
with open(f'{RESULTS_EDA}/EDA_SUMMARY_REPORT.txt', 'w') as f:
    f.write(summary_report)

print("✅ Saved: EDA_SUMMARY_REPORT.txt")
print(summary_report)

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✅ NOTEBOOK 02 (EDA) COMPLETE!")
print("="*80)

print(f"""
📊 ANALYSIS COMPLETED:
   ✓ Univariate analysis (distributions)
   ✓ Comparative analysis (cities comparison)
   ✓ Correlation analysis (heatmaps)
   ✓ Statistical summaries
   ✓ AQI categorization & compliance
   ✓ Violations analysis
   ✓ Pollutant relationships
   ✓ Meteorological impact analysis

💾 FILES SAVED ({RESULTS_EDA}/):
   ✓ 01_pollutants_distributions.html
   ✓ 02_meteorology_distributions.html
   ✓ 03_cities_pollutants_comparison.html
   ✓ 04_correlation_heatmaps.html
   ✓ 05_statistics_table.html
   ✓ 06_aqi_distribution.html
   ✓ 07_compliance_analysis.html
   ✓ 08_violations_analysis.html
   ✓ 09_pollutants_relationships.html
   ✓ 10_meteorological_impact.html
   ✓ statistics_summary.csv
   ✓ EDA_SUMMARY_REPORT.txt

🎨 VISUALIZATIONS:
   ✓ Professional color schemes (consistent & harmonious)
   ✓ Clear titles & labels
   ✓ Interactive Plotly charts
   ✓ Responsive layouts
   ✓ Clean design with good UX

🚀 NEXT PHASE:
   → Notebook 03: Time Series Analysis
   → Notebook 04: Geospatial Analysis
   → Notebook 05: Pattern Analysis
   → Notebook 06: Anomaly Detection
   → Notebook 07: Feature Engineering
   → Notebook 08: ML Forecasting

""")

print("✅ Ready for next phase!")


SECTION 8: VIOLATIONS COUNT & SEVERITY

📊 VIOLATIONS COUNT (days exceeding WHO limits):
    City  PM2.5 Violations  PM10 Violations  NO₂ Violations  O₃ Violations
  Ancona            123129             9493            8377          23456
  Athens            577586           139983          255898         219564
Zaragoza                 0             8577           16409           7684

✅ Saved: 08_violations_analysis.html



SECTION 9: POLLUTANT RELATIONSHIPS
✅ Saved: 09_pollutants_relationships.html
Buffered data was truncated after reaching the output size limit.